# 09.12 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/09-nlp-genai/12-agents-tool-calling/solutions.ipynb)

> **About the outputs in this notebook.** They were produced with `llama3.2` (3B) via Ollama on a laptop. Yours will differ: a different model, a different run, even the same model on a different machine. Small local models make more mistakes than large hosted ones; that is part of what this lesson shows.

In [1]:
# --- course files: found locally, downloaded on Colab -------------------------
import sys, urllib.request
from pathlib import Path
REPO_RAW = "https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/"

def repo_file(rel):
    """Return a local path to a file in this repo, downloading it if needed."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / rel).exists():
            return parent / rel
    dest = Path("course_files") / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

In [2]:
sys.path.insert(0, str(repo_file("lib/course_llm.py").parent))
import course_llm
from course_llm import ask, ask_json, chat, info
info();

LLM provider: ollama · model: llama3.2


In [3]:
sys.path.insert(0, str(repo_file("data/download.py").parent))
from download import load_retail
import re
sales = load_retail()                                   # ShopWave's order lines
docs_dir = repo_file("data/shopwave_docs/returns.md").parent

def order_status(invoice):
    """Look up one order by its 6-digit invoice number."""
    if not re.fullmatch(r"C?\d{6}", invoice.strip()):
        return "Error: an invoice number is 6 digits, for example 536365."
    lines = sales[sales["Invoice"] == invoice.strip()]
    if lines.empty:
        return f"No order found with invoice {invoice}."
    status = "cancelled" if invoice.startswith("C") else "delivered"
    total = (lines["Quantity"] * lines["Price"]).sum()
    return (f"Invoice {invoice}: placed {lines['InvoiceDate'].min():%d %b %Y}, {len(lines)} lines, "
            f"total £{total:,.2f}, shipped to {lines['Country'].iloc[0]}, status {status}.")

In [4]:
def refund_policy(_=""):
    """ShopWave's returns and refunds rules."""
    return ((docs_dir / "returns.md").read_text() + "\n" + (docs_dir / "refunds.md").read_text())

def product_search(query):
    """Find products whose name contains all the query words; show typical price."""
    words = query.lower().split()
    names = sales["Description"].fillna("").str.lower()
    hits = sales[names.apply(lambda n: all(w in n for w in words))]
    if hits.empty:
        return f"No products match '{query}'."
    top = hits.groupby("Description")["Price"].agg(["median", "count"]).nlargest(5, "count")
    return "; ".join(f"{d.title()} (£{r['median']:.2f})" for d, r in top.iterrows())

TOOL_FUNCTIONS = {"order_status": order_status, "refund_policy": refund_policy,
                  "product_search": product_search}

In [5]:
import json
AGENT_SYSTEM = """You are ShopWave's support agent. You can use these tools:
- order_status(argument = 6-digit invoice number): date, total, country and status of an order
- refund_policy(argument = ""): ShopWave's returns and refunds rules
- product_search(argument = a few words of a product name): matching products and typical prices
Each turn, reply with JSON: a short thought, the tool to use, and its argument.
When you have enough information, use the tool "final_answer" with the answer to the customer as the argument.
Use only facts from tool results. Never invent order details or prices."""

STEP = {"type": "object", "properties": {
            "thought": {"type": "string"},
            "tool": {"type": "string", "enum": [*TOOL_FUNCTIONS, "final_answer"]},
            "argument": {"type": "string"}},
        "required": ["thought", "tool", "argument"], "additionalProperties": False}

In [6]:
def run_agent(question, max_steps=5, verbose=True):
    messages = [{"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        reply = chat(messages, system=AGENT_SYSTEM, max_tokens=400, temperature=0, schema=STEP)
        decision = json.loads(reply)
        if verbose:
            print(f"step {step}: {decision['tool']}({decision['argument']!r})  # {decision['thought']}")
        if decision["tool"] == "final_answer":
            return decision["argument"]
        result = TOOL_FUNCTIONS[decision["tool"]](decision["argument"])
        if verbose:
            print(f"        result: {result[:150]}")
        messages += [{"role": "assistant", "content": reply},
                     {"role": "user", "content": f"Tool result: {result}"}]
    return "Sorry, I could not finish this request. A colleague will reply by email."

In [7]:
def order_status_checked(invoice):
    """Same lookup, but a bad number gets an instruction, not an example to copy."""
    if not re.fullmatch(r"C?\d{6}", invoice.strip()):
        return "No valid invoice number. Ask the customer for their 6-digit invoice number."
    return order_status(invoice)

GUARDED_TOOLS = {**TOOL_FUNCTIONS, "order_status": order_status_checked}
GUARDED_SYSTEM = AGENT_SYSTEM + """
Rules:
- Call each tool at most once with the same argument.
- If you need information the customer has not given (such as an invoice number), use final_answer to ask for it.
- Write the final answer as one or two plain sentences for the customer, never JSON."""

In [8]:
def run_guarded_agent(question, max_steps=5):
    messages, seen = [{"role": "user", "content": question}], set()
    for step in range(1, max_steps + 1):
        reply = chat(messages, system=GUARDED_SYSTEM, max_tokens=400, temperature=0, schema=STEP)
        d = json.loads(reply)
        print(f"step {step}: {d['tool']}({d['argument']!r})")
        if d["tool"] == "final_answer":
            return d["argument"]
        if (d["tool"], d["argument"]) in seen:                 # guardrail in code, not in the prompt
            result = "You already have this result. Answer the customer now with final_answer."
        else:
            seen.add((d["tool"], d["argument"]))
            result = GUARDED_TOOLS[d["tool"]](d["argument"])
        messages += [{"role": "assistant", "content": reply},
                     {"role": "user", "content": f"Tool result: {result}"}]
    return "Sorry, I could not finish this request. A colleague will reply by email."

## Exercise 1

In [9]:
def product_search_tolerant(query):
    words = [w[:-1] if w.endswith("s") and len(w) > 3 else w for w in query.lower().split()]
    if not words:
        return "Give a few words of a product name, for example 'hot water bottle'."
    return product_search(" ".join(words))

GUARDED_TOOLS["product_search"] = product_search_tolerant
print(product_search_tolerant("hot water bottles")[:120])
print("ANSWER:", run_guarded_agent("Do you sell hot water bottles? Roughly how much are they?"))

Chocolate Hot Water Bottle (£4.95); Hot Water Bottle Keep Calm (£4.95); Scottie Dog Hot Water Bottle (£4.95); Hot Water 


step 1: product_search('hot water bottles')


step 2: product_search('')


step 3: final_answer('Our hot water bottles are priced at £4.25 to £4.95, depending on the design.')
ANSWER: Our hot water bottles are priced at £4.25 to £4.95, depending on the design.


A small fix in the **tool** removes a whole class of wasted steps: the plural query now finds the products at the first attempt. The empty-query guard matters too: without it, a confused model calling `product_search("")` gets back a list of random best-sellers and may build an answer on them. When an agent flounders, look at what the tools returned before rewriting the prompt, and check the final answer: a small model can still write a muddled sentence from a correct tool result.

## Exercise 2

In [10]:
import pandas as pd
TODAY = pd.Timestamp("2011-12-10")

def return_deadline(invoice):
    lines = sales[sales["Invoice"] == invoice.strip()]
    if lines.empty:
        return "No valid order found. Ask the customer for their 6-digit invoice number."
    deadline = lines["InvoiceDate"].min().normalize() + pd.Timedelta(days=30)
    still_ok = TODAY <= deadline
    return (f"Order {invoice}: return deadline {deadline:%d %b %Y} (30 days, standard items). "
            f"Today is {TODAY:%d %b %Y}: returns {'still allowed' if still_ok else 'no longer allowed'}.")

In [11]:
GUARDED_TOOLS["return_deadline"] = return_deadline
STEP["properties"]["tool"]["enum"] = [*GUARDED_TOOLS, "final_answer"]
GUARDED_SYSTEM = GUARDED_SYSTEM.replace("- product_search", "- return_deadline(argument = 6-digit "
    "invoice number): the return deadline and whether returns are still allowed today\n- product_search")
for q in ["Order 581587: can I still return it?", "Order 536365: can I still return it?"]:
    print(f"\nQ: {q}\nANSWER:", run_guarded_agent(q))

step 1: return_deadline('581587')


step 2: order_status('581587')


step 3: refund_policy('')


step 4: final_answer('You are eligible for a refund within 30 days of delivery. However, please note that the original delivery charge will only be refunded if you return the whole order or if the item was faulty or damaged. Express delivery upgrades are never refunded unless the parcel arrived late.')

Q: Order 581587: can I still return it?
ANSWER: You are eligible for a refund within 30 days of delivery. However, please note that the original delivery charge will only be refunded if you return the whole order or if the item was faulty or damaged. Express delivery upgrades are never refunded unless the parcel arrived late.


step 1: return_deadline('536365')


step 2: refund_policy('')


step 3: final_answer('Unfortunately, the return deadline for order 536365 has passed and returns are no longer allowed. However, you can still return most items within 14 days of delivery for a refund, but not the sale items in this order.')

Q: Order 536365: can I still return it?
ANSWER: Unfortunately, the return deadline for order 536365 has passed and returns are no longer allowed. However, you can still return most items within 14 days of delivery for a refund, but not the sale items in this order.


Order 581587 was placed on 9 December 2011, so it is still inside the 30-day window on 10 December; 536365 (1 December 2010) is a year past it. The date logic now happens in Python, and the core of both answers is right. Read them in full, though: in our run the model also fetched the refund policy and padded the answers with loosely related rules (the 536365 answer muddles in the 14-day sale-item limit). The next fixes would be a rule "if `return_deadline` answered the question, reply with that", or letting code write the sentence and the model only choose the tool. (A real version would also use the delivery date rather than the order date, and the shorter limit for sale items.)

## Exercise 3

In [12]:
def run_budgeted_agent(question, budget=3000, max_steps=5):
    messages, seen, used = [{"role": "user", "content": question}], set(), 0
    for step in range(1, max_steps + 1):
        reply = chat(messages, system=GUARDED_SYSTEM, max_tokens=400, temperature=0, schema=STEP)
        used += course_llm.last_usage["input"] + course_llm.last_usage["output"]
        if used > budget:
            return f"[stopped at {used} tokens] Sorry, a colleague will reply by email.", used
        d = json.loads(reply)
        if d["tool"] == "final_answer":
            return d["argument"], used
        result = ("You already have this result." if (d["tool"], d["argument"]) in seen
                  else GUARDED_TOOLS[d["tool"]](d["argument"]))
        seen.add((d["tool"], d["argument"]))
        messages += [{"role": "assistant", "content": reply},
                     {"role": "user", "content": f"Tool result: {result}"}]
    return "Sorry, a colleague will reply by email.", used

In [13]:
QUESTIONS = [
    "What's the status of order 536365, and how much was it?",
    "Do you sell hot water bottles? Roughly how much are they?",
    "I bought a cushion three weeks ago and I don't like it. Can I get a refund?",
    "Order 536365: how much was it, and can I still return it today?",
]

In [14]:
for q in QUESTIONS:
    answer, used = run_budgeted_agent(q)
    print(f"{used:5} tokens  {q[:45]:45} → {answer[:70]}")

  671 tokens  What's the status of order 536365, and how mu → Order 536365 was delivered on 01 Dec 2010 and the total was £139.12.


 1093 tokens  Do you sell hot water bottles? Roughly how mu → Our hot water bottles are priced at £4.25 to £4.95.


 1123 tokens  I bought a cushion three weeks ago and I don' → Could you please provide the correct 6-digit invoice number for your o


  678 tokens  Order 536365: how much was it, and can I stil → You are not eligible for a return on this order as it has been shipped


Every step re-sends the whole conversation, so tokens grow faster than the number of steps: a two-step answer costs about 700 tokens, while conversations with more steps or long tool results cost noticeably more (over 1,000 tokens here). Long tool results (such as the refund policy) make any conversation that fetches them expensive. A budget turns "the agent got stuck" from a bill into a polite hand-over.

## Exercise 4

In [15]:
APPROVAL_QUEUE = []

def issue_refund(invoice):
    APPROVAL_QUEUE.append({"invoice": invoice.strip(), "requested_by": "support agent"})
    return "Refund request recorded. A colleague will review it within 1 working day."

print(issue_refund("536365"))
print(APPROVAL_QUEUE)

Refund request recorded. A colleague will review it within 1 working day.
[{'invoice': '536365', 'requested_by': 'support agent'}]


The model can be wrong, confused or manipulated (09.11's injection), so it must not be the last line of defence for anything that costs money or cannot be undone. Queueing for a person keeps the agent useful (it gathers the details and explains the next step) while a human or a rule in code makes the decision. Other ShopWave actions that should work this way: cancelling orders, changing delivery addresses, issuing vouchers, emailing customers, editing account details.